# Game of 24 — Tree of Thoughts (Yao et al., NeurIPS 2023)
**CS 4782 Final Project — Cornell University**

Replicates Table 2: IO=7.3%, CoT=4%, CoT-SC=9%, ToT-BFS=74% on GPT-4.
We run the same methods using **Qwen2.5-7B** locally via Ollama.

> Runtime → Change runtime type → **T4 GPU** before running.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q openai datasets matplotlib numpy tqdm

## Cell 2 — Install Ollama, start server, pull model

In [ ]:
import subprocess, time, urllib.request, os

# Install Ollama
os.system("sudo apt-get install -y zstd -qq")
os.system("curl -fsSL https://ollama.com/install.sh | sh")

# Kill any stale server, start fresh
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until server is ready
for i in range(20):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:11434/", timeout=2)
        print(f"Ollama server ready ({(i+1)*2}s)")
        break
    except:
        pass
else:
    print("ERROR: server did not start — check Runtime is set to GPU")

# Pull model
os.system("ollama pull qwen2.5:7b")
print("\nAvailable models:")
os.system("ollama list")


## Cell 3 — Clone repo

In [ ]:
import sys, os, subprocess

os.chdir("/content")
repo = "/content/Tree-of-Thoughts-Debugger"
if not os.path.exists(repo):
    subprocess.run(["git", "clone", "https://github.com/Datboy0127/Tree-of-Thoughts-Debugger.git"],
                   cwd="/content", check=True)
    print("Cloned.")
else:
    out = subprocess.run(["git", "pull"], cwd=repo, capture_output=True, text=True)
    print("Pulled:", out.stdout.strip() or "up to date")

sys.path.insert(0, f"{repo}/code")
os.chdir(repo)
print("Ready. CWD:", os.getcwd())


## Cell 4 — (Optional) Download paper's exact puzzle set
Problems 901-1000 from 4nums.com — the exact test set from Yao et al.
Skip this cell to use the built-in hard puzzle list instead.

In [ ]:
import urllib.request, os

csv_url = "https://raw.githubusercontent.com/princeton-nlp/tree-of-thought-llm/master/src/tot/data/24/24.csv"
os.makedirs("data", exist_ok=True)
try:
    urllib.request.urlretrieve(csv_url, "data/24.csv")
    print("Downloaded data/24.csv")
    GAME24_CSV = "data/24.csv"
except Exception as e:
    print(f"Download failed ({e}) — will use built-in puzzle list.")
    GAME24_CSV = None


## Cell 5 — Load puzzles

In [ ]:
from data_loader import load_game24

G24_N = 100   # set to 20 for a quick test

puzzles = load_game24(n=G24_N, csv_path=GAME24_CSV, seed=42)
print(f"Loaded {len(puzzles)} puzzles")
print("First 5:", puzzles[:5])


## Cell 6 — Setup LLM and solvers
Explicitly passes `model='qwen2.5:7b'` so Ollama uses the correct model.

In [ ]:
import config, os, urllib.request, subprocess, time

# Ensure Ollama server is still running before experiments
try:
    urllib.request.urlopen("http://localhost:11434/", timeout=2)
    print("Ollama server is up.")
except:
    print("Server went down — restarting...")
    subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
    time.sleep(2)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(15):
        time.sleep(2)
        try:
            urllib.request.urlopen("http://localhost:11434/", timeout=2)
            print("Server ready.")
            break
        except:
            pass

from llm_client import LLMClient
from game24_solver import (
    Game24Solver, Game24IOBaseline, Game24CoTBaseline, Game24CoTSCBaseline,
    Game24MCTSSolver, run_game24, compute_game24_metrics, print_game24_table,
)

# Explicitly specify model — avoids any config confusion
llm = LLMClient(model="qwen2.5:7b")
print(f"Model: {llm.model}  Backend: {llm.backend}  URL: {llm.base_url}")

os.makedirs("results", exist_ok=True)
g24 = {}


## Cell 7 — Run IO baseline
Paper (GPT-4): 7.3%

In [ ]:
print("── IO ──")
g24["io"] = run_game24(puzzles, Game24IOBaseline(llm), "io", verbose=True)
m = compute_game24_metrics(g24["io"])
print(f"\n  IO: {m['success_rate']:.1%}  (paper GPT-4: 7.3%)")


## Cell 8 — Run CoT baseline
Paper (GPT-4): 4%

In [ ]:
print("── CoT ──")
g24["cot"] = run_game24(puzzles, Game24CoTBaseline(llm), "cot", verbose=True)
m = compute_game24_metrics(g24["cot"])
print(f"\n  CoT: {m['success_rate']:.1%}  (paper GPT-4: 4%)")


## Cell 9 — Run CoT-SC (n=5)
Paper uses n=100 for 9%; we use n=5 to save tokens.

In [ ]:
print("── CoT-SC (n=5) ──")
g24["cot-sc-5"] = run_game24(puzzles, Game24CoTSCBaseline(llm, n_samples=5), "cot-sc-5", verbose=True)
m = compute_game24_metrics(g24["cot-sc-5"])
print(f"\n  CoT-SC (n=5): {m['success_rate']:.1%}  (paper GPT-4 n=100: 9%)")


## Cell 10 — Run ToT-BFS (b=5)
Main paper result: **74%** on GPT-4.
Algorithm 1: propose→value→keep top-b at each of 3 depth levels.

In [ ]:
print("── ToT-BFS (b=5) ──")
g24["tot-bfs-b5"] = run_game24(puzzles, Game24Solver(llm, b=5, search="bfs"), "tot-bfs-b5", verbose=True)
m = compute_game24_metrics(g24["tot-bfs-b5"])
print(f"\n  ToT-BFS: {m['success_rate']:.1%}  avg_nodes={m['avg_nodes']:.1f}  (paper GPT-4: 74%)")


## Cell 11 — Run ToT-DFS
Algorithm 2: greedy descent, prune impossible states, backtrack on dead ends.

In [ ]:
print("── ToT-DFS ──")
g24["tot-dfs-b5"] = run_game24(puzzles, Game24Solver(llm, b=5, search="dfs"), "tot-dfs-b5", verbose=True)
m = compute_game24_metrics(g24["tot-dfs-b5"])
print(f"\n  ToT-DFS: {m['success_rate']:.1%}  avg_nodes={m['avg_nodes']:.1f}")


## Cell 12 — Run MCTS (novel extension)
UCB1 over arithmetic states. LLM expands nodes; rollouts are pure math (no extra LLM calls).
Early stopping: halts as soon as any simulation finds 24.

In [ ]:
print("── MCTS (b=5, s=20) ──")
g24["mcts-b5-s20"] = run_game24(puzzles, Game24MCTSSolver(llm, b=5, n_simulations=20), "mcts-b5-s20", verbose=True)
m = compute_game24_metrics(g24["mcts-b5-s20"])
print(f"\n  MCTS: {m['success_rate']:.1%}  avg_nodes={m['avg_nodes']:.1f}")


## Cell 13 — Results table

In [ ]:
import json

print_game24_table(g24)

# Save all results
for label, results in g24.items():
    with open(f"results/game24_{label}.json", "w") as f:
        json.dump([r.__dict__ for r in results], f, indent=2)
print("\nSaved to results/")


## Cell 14 — Comparison plot: our results vs paper (GPT-4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

paper = {"IO": 7.3, "CoT": 4.0, "CoT-SC\n(n=100)": 9.0, "ToT-BFS\n(b=5)": 74.0}
key_map = {"IO": "io", "CoT": "cot", "CoT-SC\n(n=100)": "cot-sc-5", "ToT-BFS\n(b=5)": "tot-bfs-b5"}
ours = {k: compute_game24_metrics(g24.get(key_map[k], [])).get("success_rate", 0) * 100 for k in paper}

labels = list(paper.keys())
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, [paper[l] for l in labels], w, label="Paper (GPT-4)", color="#2196F3", alpha=0.85)
b2 = ax.bar(x + w/2, [ours[l] for l in labels],  w, label=f"Ours (Qwen2.5-7B)", color="#FF9800", alpha=0.85)
ax.bar_label(b1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%.1f%%", padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Success Rate (%)", fontsize=12)
ax.set_title("Game of 24 — Paper (GPT-4) vs Ours (Qwen2.5-7B)", fontsize=13)
ax.set_ylim(0, 90); ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.savefig("results/game24_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/game24_comparison.png")


## Cell 15 — Download results

In [ ]:
import shutil
shutil.make_archive("/content/game24_results", "zip", "results")
print("Zipped → /content/game24_results.zip")
try:
    from google.colab import files
    files.download("/content/game24_results.zip")
except ImportError:
    pass
